In [2]:

#Imports e seed
import numpy as np
import pandas as pd
import joblib
import json
import os
import sys

from sklearn.neighbors import NearestNeighbors

SEED = 42
np.random.seed(SEED)

print('Imports OK')

Imports OK


In [3]:
#Carregar artefatos
# Usamos o dataset COMPLETO (X inteiro, não só X_train)
# O modelo final deve conhecer todos os livros
X_npz = np.load('../data/processed/X_matrix.npz')
X     = X_npz[X_npz.files[0]]

df = pd.read_parquet('../data/processed/books_clean.parquet')
df['genre_list'] = df['genre_list'].apply(
    lambda x: list(x) if not isinstance(x, list) else x
)

print(f'X shape (dataset completo): {X.shape}')
print(f'Livros no DataFrame: {len(df)}')

X shape (dataset completo): (84054, 1182)
Livros no DataFrame: 84054


In [4]:
#Treinar o melhor modelo no dataset completo
#
# Melhor modelo identificado na comparação:
#   Algoritmo : KNN
#   Métrica   : cosine
#   k         : 20
#   Critério  : Precision@10 = 1.0000 (empatado com k=5 e k=10)
#               Coverage = 0.0071 (maior entre os empatados)
# n_neighbors = 21 (k + 1) porque precision_at_k
# chama kneighbors com n_neighbors=k+1 internamente
melhor_modelo = NearestNeighbors(
    n_neighbors=21,        # k=20 + 1
    metric='cosine',
    algorithm='brute'
)

melhor_modelo.fit(X)       # fit em 100% dos dados

print('Modelo treinado com sucesso!')
print(f'  Algoritmo : KNN')
print(f'  Métrica   : cosine')
print(f'  k         : 20')
print(f'  Livros    : {X.shape[0]}')
print(f'  Features  : {X.shape[1]}')

Modelo treinado com sucesso!
  Algoritmo : KNN
  Métrica   : cosine
  k         : 20
  Livros    : 84054
  Features  : 1182


In [5]:
#Criar pasta models/ se não existir
os.makedirs('../models', exist_ok=True)
print('Pasta ../models/ pronta')

Pasta ../models/ pronta


In [6]:
# Exportar o modelo
# Apenas o modelo vencedor vai para o .pkl
# Modelos intermediários NÃO são exportados
joblib.dump(melhor_modelo, '../models/best_model.pkl')
print('best_model.pkl salvo em ../models/')

best_model.pkl salvo em ../models/


In [7]:
#Exportar objetos auxiliares
# Esses arquivos são necessários para usar o modelo no app
# Sem eles não é possível vetorizar o input do usuário
# Carregar mlb e scaler que P1 salvou no pré-processamento
mlb    = joblib.load('../models/mlb.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Confirmar que carregaram corretamente
print(f'mlb.pkl carregado    — {len(mlb.classes_)} gêneros conhecidos')
print(f'scaler.pkl carregado — {scaler.n_features_in_} features numéricas')
print()
print('Gêneros disponíveis (primeiros 20):')
print(list(mlb.classes_[:20]))

c:\Users\conta\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MultiLabelBinarizer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


mlb.pkl carregado    — 1179 gêneros conhecidos
scaler.pkl carregado — 3 features numéricas

Gêneros disponíveis (primeiros 20):
['10th century', '11th century', '12th century', '13th century', '14th century', '15th century', '16th century', '17th century', '1864 shenandoah campaign', '18th century', '19th century', '1st grade', '20th century', '21st century', '2nd grade', '40k', 'abuse', 'academia', 'academic', 'academics']


c:\Users\conta\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
#Salvar metadados do modelo
# Documenta os hiperparâmetros e métricas para o artigo
metadata = {
    'algoritmo'           : 'KNearestNeighbors',
    'metrica'             : 'cosine',
    'k'                   : 20,
    'n_neighbors_modelo'  : 21,
    'seed'                : 42,
    'n_livros_treino'     : int(X.shape[0]),
    'n_features'          : int(X.shape[1]),
    'precision_at_10'     : 1.0000,
    'coverage'            : 0.0071,
    'criterio_selecao'    : 'Maior Coverage entre configurações com Precision@10 máxima (1.0000)',
    'arquivo_modelo'      : 'models/best_model.pkl',
    'arquivo_encoder'     : 'models/mlb.pkl',
    'arquivo_scaler'      : 'models/scaler.pkl'
}

with open('../models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('model_metadata.json salvo em ../models/')
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))

model_metadata.json salvo em ../models/

{
  "algoritmo": "KNearestNeighbors",
  "metrica": "cosine",
  "k": 20,
  "n_neighbors_modelo": 21,
  "seed": 42,
  "n_livros_treino": 84054,
  "n_features": 1182,
  "precision_at_10": 1.0,
  "coverage": 0.0071,
  "criterio_selecao": "Maior Coverage entre configurações com Precision@10 máxima (1.0000)",
  "arquivo_modelo": "models/best_model.pkl",
  "arquivo_encoder": "models/mlb.pkl",
  "arquivo_scaler": "models/scaler.pkl"
}


In [9]:
#Verificação final
# Confirma que todos os arquivos foram salvos corretamente
arquivos_esperados = [
    '../models/best_model.pkl',
    '../models/mlb.pkl',
    '../models/scaler.pkl',
    '../models/model_metadata.json'
]

print('=== Verificação de arquivos ===')
tudo_ok = True
for arq in arquivos_esperados:
    existe = os.path.exists(arq)
    status = '✓' if existe else '✗ FALTANDO'
    if not existe:
        tudo_ok = False
    tamanho = f'{os.path.getsize(arq) / 1024:.1f} KB' if existe else ''
    print(f'  {status}  {arq}  {tamanho}')

print()
if tudo_ok:
    print('Todos os arquivos presentes. Exportação concluída.')
else:
    print('ATENÇÃO: algum arquivo está faltando. Verifique as células anteriores.')

=== Verificação de arquivos ===
  ✓  ../models/best_model.pkl  776186.7 KB
  ✓  ../models/mlb.pkl  16.9 KB
  ✓  ../models/scaler.pkl  1.1 KB
  ✓  ../models/model_metadata.json  0.4 KB

Todos os arquivos presentes. Exportação concluída.


In [10]:
#Teste de sanidade
# Carrega o modelo do disco e faz uma recomendação real
# para confirmar que o .pkl está funcional
modelo_carregado = joblib.load('../models/best_model.pkl')
mlb_carregado    = joblib.load('../models/mlb.pkl')

# Simular um usuário que gosta de Fantasy e Mystery
user_genres = ['fantasy', 'mystery']

# Vetorizar o perfil do usuário
user_genre_vec = mlb_carregado.transform([user_genres])          # shape (1, n_generos)
n_numericos    = X.shape[1] - user_genre_vec.shape[1]            # colunas numéricas
user_vec       = np.hstack([user_genre_vec,
                             np.zeros((1, n_numericos))])         # completar com zeros

# Buscar vizinhos
distances, indices = modelo_carregado.kneighbors(user_vec, n_neighbors=10)

# Exibir recomendações
recomendados = df.iloc[indices[0]][['title', 'author', 'genre', 'rating']].copy()
recomendados['similaridade'] = (1 - distances[0]).round(4)

print(f'Recomendações para perfil: {user_genres}')
print()
print(recomendados.to_string(index=False))

Recomendações para perfil: ['fantasy', 'mystery']

                                                         title                      author                                                  genre  rating  similaridade
                                   The Detective & The Unicorn               Michael Angel                                Fantasy,Romance,Mystery    4.20        0.7347
ç¥žæ –éº—å¥ˆã¯æ­¤å‡¦ã«æ•£ã‚‹ [Kamisu Reina Wa Koko Ni Chiru]   Eiji Mikage,å¾¡å½± ç‘›è·¯                     Novels,Light Novel,Fantasy,Mystery    3.78        0.6614
                                  Thraxas and the Elvish Isles                Martin Scott                          Fantasy,Mystery,Humor,Fiction    4.17        0.6526
                                          The Wolves of Autumn               Scott Ciencin                                                Fantasy    2.73        0.6206
                                                Die Wolfsgrube              SzilÃ¡rd Rubin                   